# B-TRUST — Retrospective SCAR annotation-disagreement validation

**No new human raters. No GPU. No additional uploads if the Experiment B runtime is still active.**

**Start:** In the same Colab runtime as the completed eight-corpus overlap audit, run these cells. They reuse `tables`, `OUT`, and the original reconstructed data.

**Research question:** Can text-only signals predict whether the intended Bloom label differs from an existing independent expert-consensus label on SCAR questions? This evaluates **disagreement screening**, not educational correctness or a complete B-TRUST gate.

**Status:** Exploratory, retrospective, not prospective. Do not claim independent educational validation or B-TRUST improvement from this experiment.


In [ ]:
import pandas as pd, numpy as np, re, json, hashlib
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
SEED=20261121
assert 'tables' in globals() and all(x in tables for x in ('SCAR-I','SCAR-E')), (
    'Run the eight-corpus overlap audit in this SAME Colab runtime first. '
    'The five saved CSV summaries do not contain the question texts needed here.')
if 'OUT' not in globals():
    OUT=Path('/content/drive/MyDrive/BTRUST_overlap_audit')
OUT=Path(OUT); DEST=OUT/'retrospective_SCAR_validation'; DEST.mkdir(parents=True,exist_ok=True)
print('Using existing audit data; no upload required. Saving to:',DEST)


In [ ]:
# Construct one-to-one, unambiguous text-level label pairs.
# Duplicate text with multiple assigned labels is excluded rather than arbitrarily resolved.
def label_sets(df):
    return df.groupby('norm_text').label.agg(lambda x:tuple(sorted(set(x))))
a=label_sets(tables['SCAR-I']); b=label_sets(tables['SCAR-E'])
common=a.index.intersection(b.index)
pairs=pd.DataFrame({'text':common,'intended_set':a.loc[common].to_list(),'expert_set':b.loc[common].to_list()})
ambiguous=(pairs.intended_set.map(len)!=1)|(pairs.expert_set.map(len)!=1)
excluded=int(ambiguous.sum())
pairs=pairs.loc[~ambiguous].copy().reset_index(drop=True)
pairs['intended']=pairs.intended_set.map(lambda x:x[0])
pairs['expert']=pairs.expert_set.map(lambda x:x[0])
pairs['disagree']=(pairs.intended!=pairs.expert).astype(int)
assert pairs.text.is_unique and pairs.disagree.nunique()==2
print('Shared unique texts:',len(common),'Ambiguous excluded:',excluded,
      'Analyzable:',len(pairs),'Disagreements:',pairs.disagree.sum(),
      'Rate:',round(pairs.disagree.mean(),4))


In [ ]:
# 5-fold out-of-fold prediction: text-only TF-IDF versus length-only baseline.
# No source label or expert label is used as a predictor.
X=pairs.text.astype(str)
y=pairs.disagree.to_numpy()
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
text_model=make_pipeline(TfidfVectorizer(ngram_range=(1,2),min_df=3,max_features=30000,
                                         sublinear_tf=True),
                         LogisticRegression(max_iter=2000,class_weight='balanced'))
length_X=pd.DataFrame({'chars':X.str.len().astype(float),
                       'words':X.str.split().map(len).astype(float),
                       'question_mark':X.str.contains(r'\?',regex=True).astype(float)})
length_model=Pipeline([('impute',SimpleImputer(strategy='median')),
                       ('scale',StandardScaler()),
                       ('lr',LogisticRegression(max_iter=2000,class_weight='balanced'))])
# cross_val_predict ensures every score is obtained from a model that did not train on that item.
p_text=cross_val_predict(text_model,X,y,cv=cv,method='predict_proba')[:,1]
p_length=cross_val_predict(length_model,length_X,y,cv=cv,method='predict_proba')[:,1]
def report(p):
    return {'AUROC':roc_auc_score(y,p),'average_precision':average_precision_score(y,p),
            'Brier':brier_score_loss(y,p)}
results=pd.DataFrame([{'method':'TF-IDF LR (text only)',**report(p_text)},
                      {'method':'Length/format LR',**report(p_length)},
                      {'method':'Constant prevalence',**report(np.full(len(y),y.mean()))}])
display(results)


In [ ]:
# Paired bootstrap uncertainty on held-out predictions.
# Exploratory CIs; they do not account for re-fitting models in each bootstrap.
rng=np.random.default_rng(SEED)
boot=[]
for i in range(2000):
    ix=rng.integers(0,len(y),len(y))
    if len(set(y[ix]))<2: continue
    boot.append([roc_auc_score(y[ix],p_text[ix]),
                 roc_auc_score(y[ix],p_length[ix]),
                 roc_auc_score(y[ix],p_text[ix])-roc_auc_score(y[ix],p_length[ix])])
boot=np.asarray(boot)
ci=pd.DataFrame({'metric':['TF-IDF AUROC','Length AUROC','AUROC difference (TF-IDF minus length)'],
                 'estimate':[roc_auc_score(y,p_text),roc_auc_score(y,p_length),
                             roc_auc_score(y,p_text)-roc_auc_score(y,p_length)],
                 'CI_2.5%':np.percentile(boot,2.5,axis=0),
                 'CI_97.5%':np.percentile(boot,97.5,axis=0)})
display(ci)
# A fixed review budget: top 20% most suspicious texts.
budget=max(1,int(np.ceil(.20*len(y))))
def topk(p):
    idx=np.argsort(-p,kind='stable')[:budget]
    return {'review_budget':budget,'disagreements_found':int(y[idx].sum()),
            'recall_of_disagreements':float(y[idx].sum()/y.sum()),
            'precision_at_budget':float(y[idx].mean())}
screen=pd.DataFrame([{'method':'TF-IDF LR',**topk(p_text)},
                     {'method':'Length/format LR',**topk(p_length)}])
display(screen)


In [ ]:
# Save aggregate findings and hashed per-item predictions, not raw third-party question text.
hashes=pairs.text.map(lambda s:hashlib.sha256(s.encode('utf-8')).hexdigest())
pd.DataFrame({'text_sha256':hashes,'intended':pairs.intended,'expert':pairs.expert,
              'disagree':y,'p_text_oof':p_text,'p_length_oof':p_length}).to_csv(
                  DEST/'scar_hashed_oof_predictions.csv',index=False)
results.to_csv(DEST/'screening_metrics.csv',index=False)
ci.to_csv(DEST/'bootstrap_intervals.csv',index=False)
screen.to_csv(DEST/'review_budget_comparison.csv',index=False)
manifest={'experiment':'retrospective SCAR intended-vs-expert-consensus disagreement screening',
          'status':'exploratory; NOT prospective or independent educational validation',
          'n_shared_unique':int(len(common)),'n_excluded_ambiguous':excluded,
          'n_analyzable':int(len(y)),'n_disagree':int(y.sum()),
          'review_budget_fraction':0.20,'cv':'5-fold stratified out-of-fold',
          'seed':SEED,'predictors':'text only; no intended or expert labels as predictors',
          'limitation':'Expert consensus exists only for selected SCAR items; possible selection bias; no proof of Bloom construct correctness.'}
(DEST/'study_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Saved:',*[p.name for p in sorted(DEST.iterdir())],sep='\n - ')


## How to interpret

A high AUROC would mean disagreement is *predictable from wording* on this selected SCAR subset. It does **not** establish that intended or expert labels are objectively correct, that the method generalizes to other corpora, or that the full B-TRUST G0–G4 framework outperforms an alternative review policy.

If performance is weak, retain that negative result. The strongest defensible output is the paired disagreement audit and its uncertainty, not a claim of prospective educational validity.
